In [2]:


import pandas as pd
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

# =====================  =====================
csv_path = r''
duration_col = 'OSTime'
event_col    = 'OS'
exclude_cols = ['ID', 'group', duration_col, event_col]
p_thresh     = 0.05
floatfmt     = ".4f"   
# ====================================================


def df_to_md(df, floatfmt=".4f"):

    def fmt(v):
        if isinstance(v, (int, float)):
            try:
                return format(v, floatfmt)
            except Exception:
                return str(v)
        return str(v)

    headers = list(df.columns)
    lines = [
        '| ' + ' | '.join(headers) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |'
    ]
    for _, row in df.iterrows():
        lines.append('| ' + ' | '.join(fmt(v) for v in row.values) + ' |')
    return '\n'.join(lines)


def get_cindex_auto(model, data, duration_col='OSTime', event_col='OS'):

    rs = model.predict_partial_hazard(data).values.flatten()
    c_pos = concordance_index(data[duration_col], rs,  data[event_col])
    c_neg = concordance_index(data[duration_col], -rs, data[event_col])
    if c_pos >= c_neg:
        return c_pos, 
    else:
        return c_neg, 


def run_cox(df_sub, cohort_name, all_features):
    cph = CoxPHFitter()
    uni_rows = []


    for feat in all_features:
        cols_needed = [duration_col, event_col, feat]
        data = df_sub[cols_needed].dropna()
        if data.shape[0] < 5: 
            continue
        try:
            cph.fit(data, duration_col=duration_col, event_col=event_col)
            summ = cph.summary.loc[feat]
            hr = summ['exp(coef)']
            ci_low = summ['exp(coef) lower 95%']
            ci_up  = summ['exp(coef) upper 95%']
            p = summ['p']
            cidx, flag = get_cindex_auto(cph, data, duration_col, event_col)
            uni_rows.append([feat, hr, ci_low, ci_up, p, cidx, flag])
        except Exception as e:
            print(f"[{cohort_name}] {feat}: {e}")

    uni_df = pd.DataFrame(
        uni_rows,
        columns=['feature', 'HR', 'HR 95%CI lower', 'HR 95%CI upper', 'p', 'C-index', 'DirFlag']
    ).sort_values('p')


    mv_df = pd.DataFrame()
    sig_feats = uni_df[uni_df['p'] < p_thresh]['feature'].tolist()
    if sig_feats:
        mv_data = df_sub[[duration_col, event_col] + sig_feats].dropna()
        if mv_data.shape[0] >= 5:
            try:
                cph_mv = CoxPHFitter()
                cph_mv.fit(mv_data, duration_col=duration_col, event_col=event_col)
                mv_summ = cph_mv.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']]
                mv_summ = mv_summ.rename(columns={
                    'exp(coef)': 'HR',
                    'exp(coef) lower 95%': 'HR 95%CI lower',
                    'exp(coef) upper 95%': 'HR 95%CI upper'
                })
                mv_summ = mv_summ.reset_index().rename(columns={'index': 'feature'})
                mv_cidx, mv_flag = get_cindex_auto(cph_mv, mv_data, duration_col, event_col)
                mv_summ['C-index'] = mv_cidx
                mv_summ['DirFlag'] = mv_flag
                mv_df = mv_summ
            except Exception as e:
                print(f"[{cohort_name}]")


    print(f"\n### {cohort_name} ")
    if not uni_df.empty:
        print(df_to_md(uni_df, floatfmt))
    else:
        print("")

    if not mv_df.empty:
        print(f"\n### {cohort_name}")
        print(df_to_md(mv_df, floatfmt))
    else:
        print(f"\n### {cohort_name}")

    return uni_df, mv_df



if __name__ == "__main__":
    df = pd.read_csv(csv_path)


    all_features = [c for c in df.columns if c not in exclude_cols]

    # All
    uni_all, mv_all = run_cox(df, 'All', all_features)

    # Train
    df_train = df[df['group'] == 'train']
    uni_tr, mv_tr = run_cox(df_train, 'Train', all_features)

    # Val
    df_val = df[df['group'] == 'val']
    uni_val, mv_val = run_cox(df_val, 'Val', all_features)


In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.utils import median_survival_times
import numpy as np
import pandas as pd
df = pd.read_csv(r'')


def km_median_ci(time, event):

    km = KaplanMeierFitter().fit(time, event_observed=event)
    med = km.median_survival_time_
    ci_tbl = median_survival_times(km.confidence_interval_)
    return med, ci_tbl.iloc[0, 0], ci_tbl.iloc[0, 1]

def mean_sd_se(x):
    """返回 mean, sd, se"""
    x = np.asarray(x, dtype=float)
    m  = np.nanmean(x)
    sd = np.nanstd(x, ddof=1)
    se = sd / np.sqrt(np.sum(~np.isnan(x)))
    return m, sd, se

def build_summary_for_groups(df, group_col, groups, time_col='OSTime', event_col='OS',
                             overall_label='Overall'):
    rows = []

    med_os, os_l, os_h = km_median_ci(df[time_col], df[event_col])
    med_fu, fu_l, fu_h = km_median_ci(df[time_col], 1 - df[event_col])

    mean_os, sd_os, se_os = mean_sd_se(df[time_col])
    mean_fu, sd_fu, se_fu = mean_sd_se(df[time_col])  

    rows.append({
        group_col: overall_label,
        'N': len(df),
        'Events': int(df[event_col].sum()),

        # Median + 95% CI
        'Median_OS': med_os, 'OS_CI_Low': os_l, 'OS_CI_High': os_h,
        'Median_FU': med_fu, 'FU_CI_Low': fu_l, 'FU_CI_High': fu_h,

        # Mean ± SD / SE
        'Mean_OS': mean_os, 'SD_OS': sd_os, 'SE_OS': se_os,
        'Mean_FU': mean_fu, 'SD_FU': sd_fu, 'SE_FU': se_fu
    })


    for g in groups:
        sub = df[df[group_col] == g]
        if sub.empty:
            continue

        med_os, os_l, os_h = km_median_ci(sub[time_col], sub[event_col])
        med_fu, fu_l, fu_h = km_median_ci(sub[time_col], 1 - sub[event_col])

        mean_os, sd_os, se_os = mean_sd_se(sub[time_col])
        mean_fu, sd_fu, se_fu = mean_sd_se(sub[time_col])

        rows.append({
            group_col: g,
            'N': len(sub),
            'Events': int(sub[event_col].sum()),
            'Median_OS': med_os, 'OS_CI_Low': os_l, 'OS_CI_High': os_h,
            'Median_FU': med_fu, 'FU_CI_Low': fu_l, 'FU_CI_High': fu_h,
            'Mean_OS': mean_os, 'SD_OS': sd_os, 'SE_OS': se_os,
            'Mean_FU': mean_fu, 'SD_FU': sd_fu, 'SE_FU': se_fu
        })

    out_df = pd.DataFrame(rows)

    out_df[group_col] = out_df[group_col].astype(str)
    out_df = out_df.sort_values(by=group_col, key=lambda s: s.where(s != overall_label, '-1').astype(float))
    return out_df


summary_cls = build_summary_for_groups(
    df, group_col='Classification', groups=range(1, 7),
    time_col='OSTime', event_col='OS', overall_label='Overall'
)

print("\n📑 Overall + Classification 1–6 Summary")
print(summary_cls.to_string(index=False, float_format="%.2f"))


risk_group_columns = [
    '2D_RiskGroup', '2.5D_RiskGroup', '3D_RiskGroup',
    'Radiomics_RiskGroup', 'Radiologist_RiskGroup'
]
risk_rows = []
for col in risk_group_columns:
    if col not in df.columns:
        continue
    for level in ['High', 'Low']:
        sub = df[df[col] == level]
        if sub.empty:
            continue
        med_os, os_l, os_h = km_median_ci(sub['OSTime'], sub['OS'])
        med_fu, fu_l, fu_h = km_median_ci(sub['OSTime'], 1 - sub['OS'])
        mean_os, sd_os, se_os = mean_sd_se(sub['OSTime'])
        mean_fu, sd_fu, se_fu = mean_sd_se(sub['OSTime'])

        risk_rows.append({
            'Model': col,
            'Group': level,
            'N': len(sub),
            'Events': int(sub['OS'].sum()),
            'Median_OS': med_os, 'OS_CI_Low': os_l, 'OS_CI_High': os_h,
            'Median_FU': med_fu, 'FU_CI_Low': fu_l, 'FU_CI_High': fu_h,
            'Mean_OS': mean_os, 'SD_OS': sd_os, 'SE_OS': se_os,
            'Mean_FU': mean_fu, 'SD_FU': sd_fu, 'SE_FU': se_fu
        })

summary_risk = pd.DataFrame(risk_rows)
if not summary_risk.empty:
    print("\n📑 Risk Models (High/Low) Summary")
    print(summary_risk.to_string(index=False, float_format="%.2f"))